# 🎤 RVC-Python: Voice Conversion Demo

This notebook demonstrates **RVC (Retrieval-based Voice Conversion)** using the `rvc-python` library.

### What you can do:
- Upload your own audio and a pre-trained RVC model
- Convert voices with different pitch extraction methods
- Adjust pitch, protection, and other parameters
- Run everything on a free Google Colab GPU

### Requirements:
- A **.pth** RVC model file (v1 or v2)
- An audio file to convert (.wav, .mp3, .flac, etc.)
- Optional: a **.index** file for better voice similarity

---

## 1️⃣ Check GPU & Install Dependencies

Make sure you're using a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!nvidia-smi -L

In [ ]:
# Install PyTorch with CUDA 12.1 support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

# Install rvc-python from source (with Python 3.12+ support)
!pip install git+https://github.com/onxlmao/rvc-python.git -q

print("✅ Installation complete!")

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2️⃣ Upload Your Files

Upload the following files using the file browser on the left, or use the cells below:

1. **Model file** (`.pth`) — Your trained RVC model
2. **Index file** (`.index`, optional) — FAISS index for better similarity
3. **Input audio** (`.wav`, `.mp3`, etc.) — The audio you want to convert

In [ ]:
import os

# Create directories
os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("📁 Directories created:")
print("  - input/    (put your audio files here)")
print("  - output/   (converted files will appear here)")
print("  - models/   (put your .pth and .index files here)")

In [ ]:
# @title Upload Model File (.pth)
from google.colab import files
import shutil

print("📁 Select your .pth model file:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"models/{filename}")
    print(f"✅ Saved to models/{filename}")

MODEL_PATH = f"models/{list(uploaded.keys())[0]}"
print(f"\n📌 Model path: {MODEL_PATH}")

In [ ]:
# @title Upload Index File (.index) - Optional
from google.colab import files
import shutil

INDEX_PATH = ""

try:
    print("📁 Select your .index file (or cancel to skip):")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        shutil.move(filename, f"models/{filename}")
        print(f"✅ Saved to models/{filename}")
    
    INDEX_PATH = f"models/{list(uploaded.keys())[0]}"
    print(f"\n📌 Index path: {INDEX_PATH}")
except:
    print("⏭️ Skipped — no index file loaded.")

In [ ]:
# @title Upload Audio to Convert
from google.colab import files
import shutil

print("📁 Select your audio file (.wav, .mp3, etc.):")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"input/{filename}")
    print(f"✅ Saved to input/{filename}")

INPUT_PATH = f"input/{list(uploaded.keys())[0]}"
print(f"\n📌 Input path: {INPUT_PATH}")

## 3️⃣ Configure & Run Voice Conversion

Adjust the parameters below and run the conversion.

In [ ]:
# @title Conversion Settings
PITCH_SHIFT = 0          # @param {type:"slider", min:-12, max:12, step:1}
PITCH_METHOD = "rmvpe"  # @param ["rmvpe", "harvest", "crepe", "pm"]
INDEX_RATE = 0.5         # @param {type:"slider", min:0, max:1, step:0.05}
FILTER_RADIUS = 3        # @param {type:"slider", min:0, max:7, step:1}
PROTECT = 0.33           # @param {type:"slider", min:0, max:0.5, step:0.01}
RMS_MIX_RATE = 1.0       # @param {type:"slider", min:0, max:1, step:0.05}
RESAMPLE_SR = 0          # @param {type:"slider", min:0, max:48000, step:1000}
MODEL_VERSION = "v2"     # @param ["v1", "v2"]

print("⚙️ Settings:")
print(f"  Pitch shift:    {PITCH_SHIFT} semitones")
print(f"  Pitch method:   {PITCH_METHOD}")
print(f"  Index rate:     {INDEX_RATE}")
print(f"  Filter radius:  {FILTER_RADIUS}")
print(f"  Protect:        {PROTECT}")
print(f"  RMS mix rate:   {RMS_MIX_RATE}")
print(f"  Resample SR:    {RESAMPLE_SR if RESAMPLE_SR > 0 else 'original'}")
print(f"  Model version:  {MODEL_VERSION}")

In [ ]:
# @title Run Voice Conversion
import time
from rvc_python.infer import RVCInference

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"
print(f"🔧 Initializing RVC on {device}...")

t0 = time.time()
rvc = RVCInference(device=device)
print(f"✅ RVC initialized in {time.time() - t0:.1f}s")

print(f"\n📦 Loading model: {MODEL_PATH}")
t1 = time.time()
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=INDEX_PATH)
print(f"✅ Model loaded in {time.time() - t1:.1f}s")

print(f"\n⚙️ Setting parameters...")
rvc.set_params(
    f0up_key=PITCH_SHIFT,
    f0method=PITCH_METHOD,
    index_rate=INDEX_RATE,
    filter_radius=FILTER_RADIUS,
    protect=PROTECT,
    rms_mix_rate=RMS_MIX_RATE,
    resample_sr=RESAMPLE_SR,
)

input_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]
output_path = f"output/{input_name}_converted.wav"

print(f"\n🎤 Converting: {INPUT_PATH}")
print(f"   Output:   {output_path}")
t2 = time.time()
rvc.infer_file(INPUT_PATH, output_path)
elapsed = time.time() - t2
print(f"✅ Conversion complete in {elapsed:.1f}s")

# Get file size
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"📁 Output file: {output_path} ({size_mb:.1f} MB)")

## 4️⃣ Listen & Download

In [ ]:
# @title Play the converted audio
from IPython.display import Audio, display

print("🔊 Converted audio:")
display(Audio(output_path))

In [ ]:
# @title Play the original audio (for comparison)
from IPython.display import Audio, display

print("🔊 Original audio:")
display(Audio(INPUT_PATH))

In [ ]:
# @title Download converted audio
from google.colab import files

print("💾 Downloading converted audio...")
files.download(output_path)

## 5️⃣ Batch Conversion (Optional)

Upload multiple audio files to `input/` and convert them all at once.

In [ ]:
# @title Upload multiple audio files
from google.colab import files
import shutil

print("📁 Select multiple audio files to convert:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"input/{filename}")
    print(f"✅ Saved to input/{filename}")

print(f"\n📂 {len(uploaded)} file(s) uploaded")

In [ ]:
# @title Run batch conversion
import time
from rvc_python.infer import RVCInference

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"

rvc = RVCInference(device=device)
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=INDEX_PATH)
rvc.set_params(
    f0up_key=PITCH_SHIFT,
    f0method=PITCH_METHOD,
    index_rate=INDEX_RATE,
    filter_radius=FILTER_RADIUS,
    protect=PROTECT,
    rms_mix_rate=RMS_MIX_RATE,
    resample_sr=RESAMPLE_SR,
)

t0 = time.time()
results = rvc.infer_dir("input", "output")
elapsed = time.time() - t0

print(f"✅ Converted {len(results)} files in {elapsed:.1f}s")
for r in results:
    size_mb = os.path.getsize(r) / 1024 / 1024
    print(f"  📁 {r} ({size_mb:.1f} MB)")

In [ ]:
# @title Download all converted files as ZIP
import zipfile
from google.colab import files

zip_path = "converted_audio.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk("output"):
        for file in files_list:
            if file.endswith('.wav'):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, "output")
                zf.write(file_path, arcname)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"📦 Created {zip_path} ({size_mb:.1f} MB)")
files.download(zip_path)

## 6️⃣ Experiment with Different Settings

Try different pitch shifts and methods to find the best result.

In [ ]:
# @title Quick A/B comparison with different pitch shifts
from IPython.display import Audio, display
from rvc_python.infer import RVCInference
import time

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"

rvc = RVCInference(device=device)
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=INDEX_PATH)

# Convert with different pitch shifts
pitch_shifts = [-3, 0, 3, 5]
input_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]

for shift in pitch_shifts:
    rvc.set_params(f0up_key=shift, f0method=PITCH_METHOD)
    out = f"output/{input_name}_pitch{shift:+d}.wav"
    
    t0 = time.time()
    rvc.infer_file(INPUT_PATH, out)
    elapsed = time.time() - t0
    
    print(f"🎵 Pitch {shift:+d} semitones ({elapsed:.1f}s):")
    display(Audio(out))
    print()

---

## Tips

| Parameter | Recommendation |
|-----------|---------------|
| **Pitch method** | Use `rmvpe` for best quality. `crepe` can be better for clean speech but is slower. |
| **Index rate** | Set to `0.5–0.8` if you have an index file. Set to `0` to ignore the index. |
| **Protect** | Keep at `0.33–0.5` to preserve breath sounds and consonants. |
| **Pitch shift** | `+12` = one octave up, `-12` = one octave down. |
| **GPU memory** | If you get OOM errors, restart the runtime and try CPU mode (`cpu:0`). |

## Troubleshooting

- **CUDA out of memory**: Use a smaller model or switch to CPU
- **Bad audio quality**: Try a different pitch method or adjust the index rate
- **Wrong voice**: Make sure you uploaded the correct `.pth` model file
- **Install errors**: Make sure you're using a GPU runtime and re-run the install cell

## Links

- **GitHub**: [onxlmao/rvc-python](https://github.com/onxlmao/rvc-python)
- **Issues**: [Report a bug](https://github.com/onxlmao/rvc-python/issues)